In [1]:
import pandas as pd
import numpy as np


In [4]:
columns = ["Buying", "Maintenance" , "Doors" ,"Persons", "Luggage", "Safety", "Verdict"]
df = pd.read_csv("../data/car.data", header=None, names=columns)
df.head()

,Buying,Maintenance,Doors,Persons,Luggage,Safety,Verdict
0,vhigh,vhigh,2,2,small,low,unacc
1,vhigh,vhigh,2,2,small,med,unacc
2,vhigh,vhigh,2,2,small,high,unacc
3,vhigh,vhigh,2,2,med,low,unacc
4,vhigh,vhigh,2,2,med,med,unacc


In [6]:
print(df.shape)
print(df.dtypes)

(1728, 7)
Buying         str
Maintenance    str
Doors          str
Persons        str
Luggage        str
Safety         str
Verdict        str
dtype: object


## Checking Value Counts

In [7]:
for col in df.columns:
    print(f"\n{col}")
    print(df[col].value_counts())


Buying
Buying
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64

Maintenance
Maintenance
vhigh    432
high     432
med      432
low      432
Name: count, dtype: int64

Doors
Doors
2        432
3        432
4        432
5more    432
Name: count, dtype: int64

Persons
Persons
2       576
4       576
more    576
Name: count, dtype: int64

Luggage
Luggage
small    576
med      576
big      576
Name: count, dtype: int64

Safety
Safety
low     576
med     576
high    576
Name: count, dtype: int64

Verdict
Verdict
unacc    1210
acc       384
good       69
vgood      65
Name: count, dtype: int64


## Target Class is highly imbalanced. Unacceptable class is 70% which means a null classifier can achieve 70% Easily.

## Since the features are categorical, we cannot use .Corr() function to calcualte target and feature associations. We need to use Cross Tabulation in this case.

In [9]:
pd.crosstab(df["Safety"], df["Verdict"], normalize="index").round(3)

Verdict,acc,good,unacc,vgood
Safety,,,,
high,0.354,0.052,0.481,0.113
low,0.000,0.000,1.000,0.000
med,0.312,0.068,0.620,0.000


The distance betwee (low,med), (low,hig) and (med,high)

|1.000−0.620| + |0.000−0.312| + |0.000−0.068| + |0.000−0.000|
= 0.380 + 0.312 + 0.068 + 0.000 = 0.760

|1.000−0.481| + |0.000−0.354| + |0.000−0.052| + |0.000−0.113|
= 0.519 + 0.354 + 0.052 + 0.113 = 1.038

|0.620−0.481| + |0.312−0.354| + |0.068−0.052| + |0.000−0.113|
= 0.139 + 0.042 + 0.016 + 0.113 = 0.310

low ↔ med:   0.760
low ↔ high:  1.038
med ↔ high:  0.310   ← by far the smallest

Here is the thing: low and high has ditance 1.038 while low med and high has distance 0.310. The distance between 2 adjacent categories is not same. So, we need to use safety as nominal feature not a natural order. Natural order will make the distance between any two adjacent categories equal to 1.

## Chen, I did this for one feature you need to see for all the features. All you need to do is to learn what does this mean?
Hint: For low safety, the car is never accepted. For low and med safety, the acceptance was never vgood.

# To DO: Perform Cross Tabulation for all features.

In [10]:
for col in ["Buying", "Maintenance", "Doors", "Persons", "Luggage", "Safety"]:
    ct = pd.crosstab(df[col], df["Verdict"], normalize="index")
    spread = ct["unacc"].max() - ct["unacc"].min()
    print(f"{col}: unacc ranges from {ct['unacc'].min():.3f} to {ct['unacc'].max():.3f} (spread={spread:.3f})")

Buying: unacc ranges from 0.597 to 0.833 (spread=0.236)
Maintenance: unacc ranges from 0.620 to 0.833 (spread=0.213)
Doors: unacc ranges from 0.676 to 0.755 (spread=0.079)
Persons: unacc ranges from 0.542 to 1.000 (spread=0.458)
Luggage: unacc ranges from 0.639 to 0.781 (spread=0.142)
Safety: unacc ranges from 0.481 to 1.000 (spread=0.519)


If you look into the car.names dataset, it shows that there is hierarchial stuture of the dataset. It is worth checking if two features combined can do better than one alone. For example car.names states:

CAR                      car acceptability
   . PRICE                  overall price
   . . buying               buying price
   . . maint                price of the maintenance
   . TECH                   technical characteristics
   . . COMFORT              comfort
   . . . doors              number of doors
   . . . persons            capacity in terms of persons to carry
   . . . lug_boot           the size of luggage boot
   . . safety               estimated safety of the car

Combine two features to check their affect. 

In [11]:
joint_counts = pd.crosstab([df["Safety"], df["Persons"]], df["Verdict"])
joint_rates = pd.crosstab([df["Safety"], df["Persons"]], df["Verdict"], normalize="index").round(3)

print("Raw counts:")
print(joint_counts)
print()
print("Row-normalized rates:")
print(joint_rates)

Raw counts:
Verdict         acc  good  unacc  vgood
Safety Persons                         
high   2          0     0    192      0
       4        108    18     36     30
       more      96    12     49     35
low    2          0     0    192      0
       4          0     0    192      0
       more       0     0    192      0
med    2          0     0    192      0
       4         90    18     84      0
       more      90    21     81      0

Row-normalized rates:
Verdict           acc   good  unacc  vgood
Safety Persons                            
high   2        0.000  0.000  1.000  0.000
       4        0.562  0.094  0.188  0.156
       more     0.500  0.062  0.255  0.182
low    2        0.000  0.000  1.000  0.000
       4        0.000  0.000  1.000  0.000
       more     0.000  0.000  1.000  0.000
med    2        0.000  0.000  1.000  0.000
       4        0.469  0.094  0.438  0.000
       more     0.469  0.109  0.422  0.000


Chen, do you see there are 100% Unacceptabilty when there are 2 person sitting in the car. 

## FUnction for Cross checking in pairs

In [12]:
def check_combined_effect(df, feature_a, feature_b, target="Verdict"):

    counts = pd.crosstab([df[feature_a], df[feature_b]], df[target])
    rates = pd.crosstab([df[feature_a], df[feature_b]], df[target],
                         normalize="index").round(3)

    print(f"--- {feature_a} x {feature_b} vs {target} ---")
    print("Counts:")
    print(counts)
    print("\nRates:")
    print(rates)
    print()

    return counts, rates


check_combined_effect(df, "Buying", "Maintenance")   # both feed PRICE
check_combined_effect(df, "Safety", "Persons")       # both feed TECH (indirectly)
check_combined_effect(df, "Doors", "Luggage")        # both feed COMFORT

--- Buying x Maintenance vs Verdict ---
Counts:
Verdict             acc  good  unacc  vgood
Buying Maintenance                         
high   high          36     0     72      0
       low           36     0     72      0
       med           36     0     72      0
       vhigh          0     0    108      0
low    high          33     0     62     13
       low           10    23     62     13
       med           10    23     62     13
       vhigh         36     0     72      0
med    high          36     0     72      0
       low           10    23     62     13
       med           33     0     62     13
       vhigh         36     0     72      0
vhigh  high           0     0    108      0
       low           36     0     72      0
       med           36     0     72      0
       vhigh          0     0    108      0

Rates:
Verdict               acc   good  unacc  vgood
Buying Maintenance                            
high   high         0.333  0.000  0.667   0.00
       low 

(Verdict        acc  good  unacc  vgood
 Doors Luggage                         
 2     big       36     6     92     10
       med       30     6    108      0
       small     15     3    126      0
 3     big       36     6     92     10
       med       33     6    100      5
       small     30     6    108      0
 4     big       36     6     92     10
       med       36     6     92     10
       small     30     6    108      0
 5more big       36     6     92     10
       med       36     6     92     10
       small     30     6    108      0,
 Verdict          acc   good  unacc  vgood
 Doors Luggage                            
 2     big      0.250  0.042  0.639  0.069
       med      0.208  0.042  0.750  0.000
       small    0.104  0.021  0.875  0.000
 3     big      0.250  0.042  0.639  0.069
       med      0.229  0.042  0.694  0.035
       small    0.208  0.042  0.750  0.000
 4     big      0.250  0.042  0.639  0.069
       med      0.250  0.042  0.639  0.069
       sm

One reason to find pariwise cross tabulation is you can mention in report like when the safety is high and persons!= 2, Vgood was the verdict. Somehting like this. 

Next steps for you:

1- For a new query car Q, compute the combined VDM distance (sum of per-feature deltas, no root) between Q and every single row in the training set 
2- Sort all those distances, smallest to largest.
3- Take the k smallest — these are Q's k nearest neighbors.
4- Majority vote among those k neighbors' actual class labels (unacc/acc/good/vgood) — whichever label appears most often among them is your prediction for Q. If there's a tie, pick randomly (per the PDF's explicit instruction).


This EDA actually answered us why hamming distance and OHE is not good enough here. Let me know ifj you have any questions. 